In [ ]:
import os

os.environ.setdefault("HF_HOME", "/nfsd/lttm4/tesisti/shahrampour/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_transformers")

for k in ["HF_HOME","HF_DATASETS_CACHE","TRANSFORMERS_CACHE"]:
    os.makedirs(os.environ[k], exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])

## 1) Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
import copy

from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
RUN_NAME = "cifar100_5step_rank_extension_n2"

RESULTS_DIR = os.path.join("results", RUN_NAME)
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

STEP1_OUT = f"outputs/{RUN_NAME}/step1_scratch"
STEP1_FINAL_OUT = f"outputs/{RUN_NAME}/step1_scratch_final"
JOINT_OUT = f"outputs/{RUN_NAME}/joint_full"

RANK_EXT_DIR = f"outputs/{RUN_NAME}/rank_extension"
RANK_EXT_FINAL_DIR = os.path.join(RANK_EXT_DIR, "final_model")

os.makedirs(RANK_EXT_DIR, exist_ok=True)
os.makedirs(RANK_EXT_FINAL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(f"outputs/{RUN_NAME}", exist_ok=True)

## 2) Load CIFAR-100 (fine labels)

In [ ]:
from datasets import Image

dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")

dataset = dataset.cast_column("img", Image())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("example keys:", dataset["train"][0].keys())
print("first 10 classes:", class_names[:10])

In [ ]:
def make_custom_eval_dataset(class_ids, remap_labels=True):
    test_ds = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        test_ds = test_ds.map(remap)
    else:
        label2new = None
        new2orig = None

    test_ds.set_transform(preprocess_val)
    return test_ds, label2new, new2orig

## 3) Define incremental class splits (2/5/10 steps)

In [ ]:
num_steps = 5  

assert num_classes % num_steps == 0
classes_per_step = num_classes // num_steps

class_splits = [
    list(range(s * classes_per_step, (s + 1) * classes_per_step))
    for s in range(num_steps)
]

print("classes per step:", classes_per_step)
print("split sizes:", [len(x) for x in class_splits])
print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])

# num_steps = 2

# assert num_classes % num_steps == 0
# classes_per_step = num_classes // num_steps

# class_splits = [
#     list(range(0, 50)),
#     list(range(50, 100)),
# ]

# print("classes per step:", classes_per_step)
# print("split sizes:", [len(x) for x in class_splits])
# print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])
# print("step1 sample:", class_splits[1][:10], "...", class_splits[1][-3:])

In [ ]:
first_step_classes = class_splits[0]
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(class_splits[s])
all_classes = list(range(num_classes))

print("first step classes:", len(first_step_classes))
print("later step classes:", len(later_step_classes))
print("all classes:", len(all_classes))
print("first range:", first_step_classes[:5], "...", first_step_classes[-5:])
print("later range:", later_step_classes[:5], "...", later_step_classes[-5:])

# first_step_classes = class_splits[0]
# later_step_classes = []
# for s in range(1, num_steps):
#     later_step_classes.extend(class_splits[s])
# all_classes = list(range(num_classes))

# print("first step classes:", len(first_step_classes))
# print("later step classes:", len(later_step_classes))
# print("all classes:", len(all_classes))

## 4) Model + preprocessing

In [ ]:
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

from torchvision import transforms
from PIL import Image
import numpy as np
import torch

size = image_processor.size
if isinstance(size, dict):
    H = size.get("height", 224)
    W = size.get("width", 224)
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H + 32, W + 32)),
    transforms.RandomCrop((H, W)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = x.astype(np.uint8)
        arr = np.squeeze(arr)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if not (arr.ndim == 3 and arr.shape[-1] in (1, 3)):
            raise TypeError(f"Unexpected image array shape after squeeze: {arr.shape}")

        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean().item()}

## 5) Build per-step datasets (step / cumulative / full)

In [ ]:
def classes_for_step(step_idx: int) -> list[int]:
    return class_splits[step_idx]

def classes_for_cumulative(step_idx: int) -> list[int]:
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def filter_by_classes(ds, class_ids):
    class_set = set(class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda x: int(x["label"]) in class_set)

def make_step_datasets(step_idx: int, split_type: str = "new_only", remap_labels: bool = False):
    """
    split_type:
      - 'new_only'   : only classes of this step
      - 'cumulative' : union of classes up to this step
      - 'full'       : all classes (100)
    """
    if split_type == "full":
        class_ids = list(range(num_classes))
    elif split_type == "cumulative":
        class_ids = classes_for_cumulative(step_idx)
    elif split_type == "new_only":
        class_ids = classes_for_step(step_idx)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")

    train_ds = filter_by_classes(dataset["train"], class_ids)
    test_ds  = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        train_ds = train_ds.map(remap)
        test_ds  = test_ds.map(remap)
    else:
        label2new = {c: c for c in class_ids}
        new2orig = {c: c for c in class_ids}

    train_ds = train_ds.with_transform(preprocess_train)
    test_ds = test_ds.with_transform(preprocess_val)

    print(f"[Dataset] Step {step_idx+1} | split_type={split_type}")
    print(f"[Dataset] Classes: {class_ids[:5]} ... {class_ids[-5:]}")
    print(f"[Dataset] Train size: {len(train_ds)} | Test size: {len(test_ds)}")

    return train_ds, test_ds, label2new, new2orig, class_ids

eval_all = dataset["test"].with_transform(preprocess_val)

print("eval_all size:", len(eval_all))

## 6) Training recipes (reasonable settings)

In [ ]:
set_seed(42)

# SCRATCH_EPOCHS = 20
# LORA_EPOCHS    = 10
# FT_EPOCHS      = 4
# JOINT_EPOCHS   = 10

SCRATCH_EPOCHS = 20
LORA_EPOCHS    = 7
FT_EPOCHS      = 5
JOINT_EPOCHS   = 10

LR_SCRATCH = 5e-5
LR_LORA    = 7e-5
LR_FT      = 3e-5
LR_JOINT   = 5e-5

BATCH_SCRATCH = 8
ACCUM_SCRATCH = 2

BATCH_LORA    = 16
ACCUM_LORA    = 1

BATCH_FT      = 8
ACCUM_FT      = 2

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"
USE_FP16 = torch.cuda.is_available()

TARGET_MODULES = ["query", "key", "value"]
LORA_DROPOUT = 0.05

USE_ORTHO_RUNS = [False, True]
LAMBDA_ORTHO = 2.0
ORTHO_LOG_EVERY = 20

NEW_RANK_PER_STEP = [12, 12, 12, 12]
ALPHA_PER_STEP    = [6, 12, 18, 24]

results = []

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "model_checkpoint": model_checkpoint,
    "num_steps": num_steps,
    "classes_per_step": classes_per_step,
    "init_mode": "scratch",
    "scratch_epochs": SCRATCH_EPOCHS,
    "lora_epochs": LORA_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "joint_epochs": JOINT_EPOCHS,
    "lr_scratch": LR_SCRATCH,
    "lr_lora": LR_LORA,
    "lr_ft": LR_FT,
    "lr_joint": LR_JOINT,
    "batch_scratch": BATCH_SCRATCH,
    "accum_scratch": ACCUM_SCRATCH,
    "batch_lora": BATCH_LORA,
    "accum_lora": ACCUM_LORA,
    "batch_ft": BATCH_FT,
    "accum_ft": ACCUM_FT,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "scheduler": SCHED,
    "target_modules": TARGET_MODULES,
    "lora_dropout": LORA_DROPOUT,
    "use_ortho_runs": USE_ORTHO_RUNS,
    "lambda_ortho": LAMBDA_ORTHO,
    "ortho_log_every": ORTHO_LOG_EVERY,
    "new_rank_per_step": NEW_RANK_PER_STEP,
    "alpha_per_step": ALPHA_PER_STEP,
    "rank_extension_mode": True,
}
print(json.dumps(run_config, indent=2))

## 7) Step 1: train full ViT from scratch on step 0 classes

In [ ]:
import os, shutil, json


if os.path.exists(STEP1_OUT):
    shutil.rmtree(STEP1_OUT)
if os.path.exists(STEP1_FINAL_OUT):
    shutil.rmtree(STEP1_FINAL_OUT)

os.makedirs(STEP1_OUT, exist_ok=True)
os.makedirs(STEP1_FINAL_OUT, exist_ok=True)

step1_idx = 0
train_step1, test_step1, label2new_1, new2orig_1, class_ids_1 = make_step_datasets(
    step1_idx, split_type="new_only", remap_labels=False
)

print("Step1 original classes:", class_ids_1[:5], "...", class_ids_1[-5:])
print(
    "Step1 label range:",
    min(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
    max(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
)

config_step1 = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

model_step1 = AutoModelForImageClassification.from_config(config_step1)

print("Before training - Step1 model num_labels:", model_step1.config.num_labels)
print("Before training - Step1 classifier out_features:", model_step1.classifier.out_features)
assert model_step1.config.num_labels == num_classes
assert model_step1.classifier.out_features == num_classes

args_step1 = TrainingArguments(
    output_dir=STEP1_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    num_train_epochs=SCRATCH_EPOCHS,
    learning_rate=LR_SCRATCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_SCRATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_SCRATCH,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
    max_grad_norm=1.0,
    label_smoothing_factor=0.1,
)

trainer_step1 = Trainer(
    model=model_step1,
    args=args_step1,
    train_dataset=train_step1,
    eval_dataset=test_step1,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_step1.remove_callback(NotebookProgressCallback)

trainer_step1.train()

eval_step1 = trainer_step1.evaluate()
print({"eval_step1": eval_step1})

print("About to save B0 final model to:", STEP1_FINAL_OUT)
print("Trainer model type:", type(trainer_step1.model))
print("Trainer model class:", trainer_step1.model.__class__.__name__)

trainer_step1.model.save_pretrained(STEP1_FINAL_OUT, safe_serialization=False)
image_processor.save_pretrained(STEP1_FINAL_OUT)

print("Saved STEP1_FINAL_OUT to:", STEP1_FINAL_OUT)
print("STEP1_FINAL_OUT exists:", os.path.exists(STEP1_FINAL_OUT))
print("Files in STEP1_FINAL_OUT:", os.listdir(STEP1_FINAL_OUT) if os.path.exists(STEP1_FINAL_OUT) else "MISSING")

cfg_path = os.path.join(STEP1_FINAL_OUT, "config.json")
print("config exists:", os.path.exists(cfg_path))

if os.path.exists(cfg_path):
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    print("model_type:", cfg.get("model_type"))
    print("architectures:", cfg.get("architectures"))
    print("num_labels:", cfg.get("num_labels"))

reloaded_step1 = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)
print("Reloaded STEP1 num_labels:", reloaded_step1.config.num_labels)
print("Reloaded STEP1 classifier out_features:", reloaded_step1.classifier.out_features)

assert reloaded_step1.config.num_labels == num_classes
assert reloaded_step1.classifier.out_features == num_classes

In [ ]:
print("Init mode:", run_config["init_mode"])
assert run_config["init_mode"] == "scratch"

In [ ]:
step1_log_df = pd.DataFrame(trainer_step1.state.log_history)
step1_log_df.to_csv(os.path.join(TABLES_DIR, "step1_log_history.csv"), index=False)

step1_metrics = {
    "experiment": "step1_scratch",
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "step1_metrics.json"), "w") as f:
    json.dump(step1_metrics, f, indent=2)

results.append({
    "experiment": "step1_scratch",
    "method": "step1_scratch",
    "step": 1,
    "eval_type": "current_step",
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
})

step1_log_df.tail()

In [ ]:
if "loss" in step1_log_df.columns:
    train_loss_df = step1_log_df[step1_log_df["loss"].notna()]
    if not train_loss_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(train_loss_df["step"], train_loss_df["loss"])
        plt.xlabel("Step")
        plt.ylabel("Train Loss")
        plt.title("Step 1 Train Loss")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_train_loss.png"), dpi=200)
        plt.show()

if "eval_accuracy" in step1_log_df.columns:
    eval_df = step1_log_df[step1_log_df["eval_accuracy"].notna()]
    if not eval_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(eval_df["epoch"], eval_df["eval_accuracy"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Eval Accuracy")
        plt.title("Step 1 Eval Accuracy")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_eval_accuracy.png"), dpi=200)
        plt.show()

## 8) Rank-Extension LoRA (grow one LoRA across steps, freeze old blocks)

In [ ]:
first_step_classes = classes_for_step(0)

def make_eval_dataset(class_ids, name=None):
    test_ds = filter_by_classes(dataset["test"], class_ids)
    test_ds = test_ds.with_transform(preprocess_val)
    if name is not None:
        print(f"[Eval set] {name}: num_classes={len(class_ids)}, size={len(test_ds)}")
    return test_ds

In [ ]:
class ExtensibleLoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, dropout: float = 0.0):
        super().__init__()
        self.base = copy.deepcopy(base_linear)
        for p in self.base.parameters():
            p.requires_grad = False

        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.dropout = nn.Dropout(dropout)

        # old copied prefix (frozen)
        self.register_buffer("A_frozen", torch.zeros(0, self.in_features))
        self.register_buffer("B_frozen", torch.zeros(self.out_features, 0))

        # only the NEW suffix stays trainable
        self.lora_As = nn.ParameterList()
        self.lora_Bs = nn.ParameterList()
        self.scalings = []

        self.prev_weight = None

    def frozen_rank(self):
        return int(self.A_frozen.shape[0])

    def trainable_rank(self):
        if len(self.lora_As) == 0:
            return 0
        return int(self.lora_As[0].shape[0])

    def current_rank(self):
        return self.frozen_rank() + self.trainable_rank()

    def num_blocks(self):
        return 1 if self.current_rank() > 0 else 0

    def freeze_all_blocks(self):
        for A in self.lora_As:
            A.requires_grad = False
        for B in self.lora_Bs:
            B.requires_grad = False

    def freeze_old_blocks(self):
        # old part is already stored in buffers
        pass

    def _zero_delta(self):
        return torch.zeros_like(self.base.weight)

    def _current_scaling(self):
        if len(self.scalings) == 0:
            return 1.0
        return float(self.scalings[0])

    def _frozen_delta(self):
        if self.A_frozen.numel() == 0 or self.B_frozen.numel() == 0:
            return self._zero_delta()
        s = self._current_scaling()
        return s * (self.B_frozen @ self.A_frozen)

    def _trainable_delta(self):
        if len(self.lora_As) == 0:
            return self._zero_delta()
        A = self.lora_As[0]
        B = self.lora_Bs[0]
        s = self._current_scaling()
        return s * (B @ A)

    def get_delta(self, idx: int):
        # compatibility method
        if idx != 0:
            raise IndexError("Only one active trainable block exists in this implementation.")
        return self.current_delta()

    def current_delta(self):
        # current step contribution only
        return self._trainable_delta()

    def total_delta(self):
        # full single growing LoRA = frozen copied prefix + trainable suffix
        return self._frozen_delta() + self._trainable_delta()

    def current_effective_weight(self):
        return self.base.weight + self.total_delta()

    def set_prev_weight_reference_from_effective(self):
        self.prev_weight = self.current_effective_weight().detach().clone()

    def set_prev_weight_reference_from_base(self):
        self.prev_weight = self.base.weight.detach().clone()

    def clear_lora_blocks(self):
        device = self.base.weight.device
        dtype = self.base.weight.dtype

        self.A_frozen = torch.zeros(0, self.in_features, device=device, dtype=dtype)
        self.B_frozen = torch.zeros(self.out_features, 0, device=device, dtype=dtype)

        self.lora_As = nn.ParameterList()
        self.lora_Bs = nn.ParameterList()
        self.scalings = []

    def add_rank_block(self, r: int, alpha: float, trainable: bool = True):
        """
        True rank-growing LoRA:
        - old full LoRA is copied into frozen prefix
        - exact old contribution is preserved under new scaling
        - only the new suffix is trainable
        """
        if r <= 0:
            raise ValueError(f"Rank increment must be positive, got r={r}")

        device = self.base.weight.device
        dtype = self.base.weight.dtype

        old_rank = self.current_rank()
        old_scaling = self._current_scaling() if old_rank > 0 else None

        if old_rank > 0:
            old_A_parts = []
            old_B_parts = []

            if self.A_frozen.numel() > 0:
                old_A_parts.append(self.A_frozen.detach().clone())
                old_B_parts.append(self.B_frozen.detach().clone())

            if len(self.lora_As) > 0:
                old_A_parts.append(self.lora_As[0].detach().clone())
                old_B_parts.append(self.lora_Bs[0].detach().clone())

            old_A = torch.cat(old_A_parts, dim=0)
            old_B = torch.cat(old_B_parts, dim=1)
        else:
            old_A = torch.zeros(0, self.in_features, device=device, dtype=dtype)
            old_B = torch.zeros(self.out_features, 0, device=device, dtype=dtype)

        new_total_rank = old_rank + r
        new_scaling = float(alpha) / float(new_total_rank)

        self.lora_As = nn.ParameterList()
        self.lora_Bs = nn.ParameterList()
        self.scalings = [new_scaling]

        if old_rank > 0:
            if abs(new_scaling) < 1e-12:
                raise ValueError("New scaling is too close to zero.")

            # preserve old delta exactly:
            # old_scaling * (old_B @ old_A) == new_scaling * (B_frozen @ A_frozen)
            scale_fix = float(old_scaling) / float(new_scaling)

            self.A_frozen = (old_A * scale_fix).to(device=device, dtype=dtype)
            self.B_frozen = old_B.to(device=device, dtype=dtype)
        else:
            self.A_frozen = torch.zeros(0, self.in_features, device=device, dtype=dtype)
            self.B_frozen = torch.zeros(self.out_features, 0, device=device, dtype=dtype)

        A_new = nn.Parameter(torch.empty(r, self.in_features, device=device, dtype=dtype))
        B_new = nn.Parameter(torch.zeros(self.out_features, r, device=device, dtype=dtype))
        nn.init.kaiming_uniform_(A_new, a=np.sqrt(5))

        A_new.requires_grad = trainable
        B_new.requires_grad = trainable

        self.lora_As.append(A_new)
        self.lora_Bs.append(B_new)

    def absorb_all_lora_into_base(self):
        merged_weight = self.current_effective_weight().detach()
        self.base.weight.data.copy_(
            merged_weight.to(self.base.weight.device, dtype=self.base.weight.dtype)
        )
        self.clear_lora_blocks()

    def forward(self, x):
        out = self.base(x)
        x_d = self.dropout(x)

        if self.A_frozen.numel() > 0:
            s = self._current_scaling()
            out = out + s * F.linear(F.linear(x_d, self.A_frozen), self.B_frozen)

        if len(self.lora_As) > 0:
            A = self.lora_As[0]
            B = self.lora_Bs[0]
            s = self._current_scaling()
            out = out + s * F.linear(F.linear(x_d, A), B)

        return out

    def merged_linear(self):
        merged = copy.deepcopy(self.base)
        merged.weight.data.copy_(
            self.current_effective_weight().detach().to(
                merged.weight.device, dtype=merged.weight.dtype
            )
        )
        return merged


def set_submodule_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new_module)


def wrap_rank_extension_modules(model, target_module_names=("query", "value"), dropout=0.0):
    replace_list = []

    for name, module in model.named_modules():
        leaf = name.split(".")[-1]
        if leaf in target_module_names and isinstance(module, nn.Linear):
            replace_list.append((name, module))

    for name, old_module in replace_list:
        new_module = ExtensibleLoRALinear(old_module, dropout=dropout)
        set_submodule_by_name(model, name, new_module)

    return model


def prepare_model_for_new_rank_step(model, r, alpha, absorb_before_new_step=False):
    """
    RankExt:
        grow one single LoRA by r, copy old learned part into frozen prefix,
        train only the new suffix

    RankExt+Ortho:
        absorb previous LoRA into base, keep prev_weight reference,
        then train a fresh rank-r LoRA
    """
    count = 0

    for module in model.modules():
        if not isinstance(module, ExtensibleLoRALinear):
            continue

        old_rank = module.current_rank()

        if absorb_before_new_step:
            if old_rank > 0:
                module.absorb_all_lora_into_base()
            module.set_prev_weight_reference_from_base()
            module.add_rank_block(r=r, alpha=alpha, trainable=True)

            print(
                f"[prepare] ORTHO | old_rank={old_rank} | "
                f"new_rank={module.current_rank()} | alpha={alpha}"
            )
        else:
            module.set_prev_weight_reference_from_effective()
            module.add_rank_block(r=r, alpha=alpha, trainable=True)

            print(
                f"[prepare] RANK-EXT | old_rank={old_rank} | "
                f"added_rank={r} | new_total_rank={module.current_rank()} | alpha={alpha}"
            )

        count += 1

    mode = "ORTHO(absorb previous)" if absorb_before_new_step else "RANK-EXT(true growing single LoRA)"
    print(f"Prepared new rank step on {count} wrapped modules | mode={mode}")


def freeze_everything_except_current_rank_block_and_classifier(model, train_classifier=True):
    for p in model.parameters():
        p.requires_grad = False

    for module in model.modules():
        if isinstance(module, ExtensibleLoRALinear):
            for p in module.base.parameters():
                p.requires_grad = False

            if len(module.lora_As) > 0:
                module.lora_As[0].requires_grad = True
                module.lora_Bs[0].requires_grad = True

    if train_classifier:
        for p in model.classifier.parameters():
            p.requires_grad = True


def count_trainable_params(model, print_names=False):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")

    if print_names:
        print("\nTrainable parameter names:")
        for n, p in model.named_parameters():
            if p.requires_grad:
                print(n)


def absorb_rank_extension_into_backbone(model):
    for module in model.modules():
        if isinstance(module, ExtensibleLoRALinear):
            module.absorb_all_lora_into_base()
    return model


def register_classifier_row_gradient_mask(classifier, trainable_rows):
    handles = []
    row_idx = torch.tensor(sorted(trainable_rows), dtype=torch.long)

    def mask_weight_grad(grad):
        idx = row_idx.to(grad.device)
        mask = torch.zeros_like(grad)
        mask[idx] = 1.0
        return grad * mask

    handles.append(classifier.weight.register_hook(mask_weight_grad))

    if classifier.bias is not None:
        def mask_bias_grad(grad):
            idx = row_idx.to(grad.device)
            mask = torch.zeros_like(grad)
            mask[idx] = 1.0
            return grad * mask
        handles.append(classifier.bias.register_hook(mask_bias_grad))

    return handles

def verify_classifier_grad_mask(classifier, current_class_ids):
    """
    Check that only current_class_ids rows have non-zero gradients.
    Must be called AFTER backward().
    """
    if classifier.weight.grad is None:
        print("No gradients found on classifier.weight")
        return

    grad = classifier.weight.grad.detach()

    current_class_ids = set(current_class_ids)

    nonzero_rows = (grad.abs().sum(dim=1) > 0).nonzero(as_tuple=True)[0].tolist()

    wrong_rows = [r for r in nonzero_rows if r not in current_class_ids]

    print(f"Trainable rows expected: {sorted(current_class_ids)[:10]} ...")
    print(f"Rows with non-zero grad: {nonzero_rows[:10]} ...")

    if len(wrong_rows) == 0:
        print("✅ Grad mask OK")
    else:
        print(f"❌ Grad leakage in rows: {wrong_rows[:10]}")


def inspect_classifier_row_status(classifier, current_class_ids, num_show=12):
    current_class_ids = sorted(current_class_ids)
    print(f"Classifier trainable rows sample: {current_class_ids[:num_show]}")


def orthogonality_loss(model, lambda_ortho=2.0):
    """
    lambda * mean_over_matrices( tr(M_{t-1} L_t^T) )
    Here:
      M_{t-1} -> prev_weight
      L_t     -> current_delta
    We square the trace-like term for a non-negative penalty.
    """
    penalties = []

    for module in model.modules():
        if not isinstance(module, ExtensibleLoRALinear):
            continue

        if module.prev_weight is None:
            continue

        if module.num_blocks() == 0:
            continue

        w_old = module.prev_weight.to(module.base.weight.device, dtype=module.base.weight.dtype)
        delta_new = module.current_delta()

        trace_term = (w_old * delta_new).sum() / w_old.numel()
        penalties.append(trace_term.pow(2))

    if len(penalties) == 0:
        return torch.tensor(0.0, device=next(model.parameters()).device)

    return lambda_ortho * torch.stack(penalties).mean()


class OrthoLoRATrainer(Trainer):
    def __init__(self, *args, use_ortho=False, lambda_ortho=2.0, ortho_log_every=20, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_ortho = use_ortho
        self.lambda_ortho = lambda_ortho
        self.ortho_log_every = ortho_log_every

    def create_optimizer(self):
        if self.optimizer is None:
            decay_params = []
            no_decay_params = []

            for n, p in self.model.named_parameters():
                if not p.requires_grad:
                    continue

                if n.startswith("classifier."):
                    no_decay_params.append(p)
                elif n.endswith(".bias") or "LayerNorm.weight" in n or "layernorm" in n.lower():
                    no_decay_params.append(p)
                else:
                    decay_params.append(p)

            optimizer_grouped_parameters = [
                {"params": decay_params, "weight_decay": self.args.weight_decay},
                {"params": no_decay_params, "weight_decay": 0.0},
            ]

            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(0.9, 0.999),
                eps=1e-8,
            )
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        ce_loss = F.cross_entropy(outputs.logits, labels)

        ortho = torch.tensor(0.0, device=ce_loss.device)
        if self.use_ortho:
            ortho = orthogonality_loss(model, lambda_ortho=self.lambda_ortho)

        loss = ce_loss + ortho

        if model.training and (self.state.global_step % self.ortho_log_every == 0):
            print({
                "step": int(self.state.global_step),
                "ce_loss": float(ce_loss.detach().cpu()),
                "ortho_loss": float(ortho.detach().cpu()),
                "total_loss": float(loss.detach().cpu()),
            })

        return (loss, outputs) if return_outputs else loss


def split_train_into_train_val(dataset, val_ratio=0.1, seed=42):
    n = len(dataset)
    if n == 0:
        raise ValueError("Dataset is empty.")

    indices = np.arange(n)

    rng = np.random.default_rng(seed)
    rng.shuffle(indices)

    n_val = max(1, int(n * val_ratio))
    if n_val >= n:
        n_val = max(1, n - 1)

    val_idx = indices[:n_val].tolist()
    train_idx = indices[n_val:].tolist()

    train_subset = dataset.select(train_idx)
    val_subset = dataset.select(val_idx)

    return train_subset, val_subset

In [ ]:
rankext_results = []
rankext_final_evals = {}

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

for use_ortho in USE_ORTHO_RUNS:
    method_name = "RankExt+Ortho" if use_ortho else "RankExt"
    exp_tag = "rankext_ortho" if use_ortho else "rankext_no_ortho"

    mode_train_root = os.path.join(RANK_EXT_DIR, exp_tag)
    mode_final_dir  = os.path.join(RANK_EXT_FINAL_DIR, exp_tag)

    if os.path.exists(mode_train_root):
        shutil.rmtree(mode_train_root)
    if os.path.exists(mode_final_dir):
        shutil.rmtree(mode_final_dir)

    os.makedirs(mode_train_root, exist_ok=True)
    os.makedirs(mode_final_dir, exist_ok=True)

    rankext_model = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)
    rankext_model = wrap_rank_extension_modules(
        rankext_model,
        target_module_names=TARGET_MODULES,
        dropout=LORA_DROPOUT,
    )

    print(f"\n{'='*80}")
    print(f"[Rank-Extension] Starting mode: {method_name}")
    print(f"{'='*80}")

    print("Model num_labels:", rankext_model.config.num_labels)
    print("Classifier out_features:", rankext_model.classifier.out_features)

    assert rankext_model.config.num_labels == num_classes
    assert rankext_model.classifier.out_features == num_classes

    for step_idx in range(1, num_steps):
        train_step_full, test_step, label2new, new2orig, class_ids = make_step_datasets(
            step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        train_step, val_step = split_train_into_train_val(
            train_step_full,
            val_ratio=0.1,
            seed=42 + step_idx,
        )

        print(f"\n[{method_name}] Step {step_idx+1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

        tr_min, tr_max = _label_range(train_step)
        va_min, va_max = _label_range(val_step)
        te_min, te_max = _label_range(test_step)

        print("Train label range:", tr_min, tr_max)
        print("Val label range  :", va_min, va_max)
        print("Test label range :", te_min, te_max)

        assert tr_min >= 0 and tr_max < rankext_model.classifier.out_features
        assert va_min >= 0 and va_max < rankext_model.classifier.out_features
        assert te_min >= 0 and te_max < rankext_model.classifier.out_features

        r_new = NEW_RANK_PER_STEP[step_idx - 1]
        alpha_new = ALPHA_PER_STEP[step_idx - 1]

        cumulative_rank_target = sum(NEW_RANK_PER_STEP[:step_idx])
        print("Added rank this step:", r_new)
        print("Cumulative rank target:", cumulative_rank_target)

        prepare_model_for_new_rank_step(
            rankext_model,
            r=r_new,
            alpha=alpha_new,
            absorb_before_new_step=use_ortho,
        )

        freeze_everything_except_current_rank_block_and_classifier(
            rankext_model,
            train_classifier=True,
        )
        inspect_classifier_row_status(rankext_model.classifier, class_ids)

        handles = register_classifier_row_gradient_mask(
            rankext_model.classifier,
            trainable_rows=class_ids,
        )

        count_trainable_params(rankext_model, print_names=(step_idx == 1))

        step_rankext_train_dir = os.path.join(mode_train_root, f"step_{step_idx+1}_train")

        args_rankext = TrainingArguments(
            output_dir=step_rankext_train_dir,
            remove_unused_columns=False,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            num_train_epochs=LORA_EPOCHS,
            learning_rate=LR_LORA,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            lr_scheduler_type=SCHED,
            per_device_train_batch_size=BATCH_LORA,
            per_device_eval_batch_size=32,
            gradient_accumulation_steps=ACCUM_LORA,
            fp16=USE_FP16,
            dataloader_num_workers=0,
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            greater_is_better=True,
            report_to="none",
            disable_tqdm=True,
            max_grad_norm=1.0,
            label_smoothing_factor=0.05,
        )

        trainer_rankext = OrthoLoRATrainer(
            model=rankext_model,
            args=args_rankext,
            train_dataset=train_step,
            eval_dataset=val_step,
            data_collator=collate_fn,
            compute_metrics=compute_metrics,
            use_ortho=use_ortho,
            lambda_ortho=LAMBDA_ORTHO,
            ortho_log_every=ORTHO_LOG_EVERY,
        )

        trainer_rankext.train()

        pd.DataFrame(trainer_rankext.state.log_history).to_csv(
            os.path.join(TABLES_DIR, f"{exp_tag}_step{step_idx+1}_log_history.csv"),
            index=False
        )

        verify_classifier_grad_mask(rankext_model.classifier, class_ids)

        for h in handles:
            h.remove()

        eval_current = trainer_rankext.evaluate(eval_dataset=test_step)

        eval_first = make_eval_dataset(classes_for_step(0), name="first_step")
        seen_now = classes_for_cumulative(step_idx)
        later_now = [c for c in seen_now if c not in classes_for_step(0)]

        eval_later = make_eval_dataset(later_now, name="later_steps") if len(later_now) > 0 else None
        eval_seen = make_eval_dataset(seen_now, name="all_seen")

        eval_first_res = trainer_rankext.evaluate(eval_dataset=eval_first)
        eval_later_res = (
            trainer_rankext.evaluate(eval_dataset=eval_later)
            if eval_later is not None
            else {"eval_accuracy": np.nan, "eval_loss": np.nan}
        )
        eval_seen_res = trainer_rankext.evaluate(eval_dataset=eval_seen)

        print(f"[{method_name}] Step {step_idx+1} | current_step(test):", eval_current)
        print(f"[{method_name}] Step {step_idx+1} | first_step       :", eval_first_res)
        print(f"[{method_name}] Step {step_idx+1} | later_steps      :", eval_later_res)
        print(f"[{method_name}] Step {step_idx+1} | all_seen         :", eval_seen_res)

        # sanity-check
        calc_seen = 0.2 * float(eval_first_res.get("eval_accuracy", np.nan)) + 0.8 * float(eval_later_res.get("eval_accuracy", np.nan))
        print(
            f"[{method_name}] Step {step_idx+1} sanity | "
            f"all_seen={float(eval_seen_res.get('eval_accuracy', np.nan)):.4f} | "
            f"recomputed≈{calc_seen:.4f}"
        )

        step_log_df = pd.DataFrame(trainer_rankext.state.log_history)
        step_log_df.to_csv(
            os.path.join(TABLES_DIR, f"{exp_tag}_step_{step_idx+1}_log.csv"),
            index=False
        )

        rankext_results.extend([
            {
                "experiment": f"{exp_tag}_step_eval",
                "method": method_name,
                "step": step_idx + 1,
                "eval_type": "current_step",
                "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_current.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"{exp_tag}_step_eval",
                "method": method_name,
                "step": step_idx + 1,
                "eval_type": "first_step",
                "eval_accuracy": float(eval_first_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_first_res.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"{exp_tag}_step_eval",
                "method": method_name,
                "step": step_idx + 1,
                "eval_type": "later_steps",
                "eval_accuracy": float(eval_later_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_later_res.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"{exp_tag}_step_eval",
                "method": method_name,
                "step": step_idx + 1,
                "eval_type": "all_seen",
                "eval_accuracy": float(eval_seen_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_seen_res.get("eval_loss", np.nan)),
            },
        ])


        if use_ortho:
            absorb_rank_extension_into_backbone(rankext_model)

    final_rankext_model = copy.deepcopy(rankext_model)

    absorb_rank_extension_into_backbone(final_rankext_model)

    final_rankext_model.save_pretrained(mode_final_dir, safe_serialization=False)
    image_processor.save_pretrained(mode_final_dir)

    args_rankext_eval = TrainingArguments(
        output_dir=os.path.join(mode_train_root, "final_eval"),
        remove_unused_columns=False,
        report_to="none",
        fp16=USE_FP16,
        per_device_eval_batch_size=32,
        dataloader_num_workers=0,
    )

    trainer_rankext_final = Trainer(
        model=final_rankext_model,
        args=args_rankext_eval,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    later_step_classes = []
    for s in range(1, num_steps):
        later_step_classes.extend(classes_for_step(s))

    all_seen_classes = classes_for_cumulative(num_steps - 1)

    rankext_test_first = make_eval_dataset(first_step_classes, name="final_first_step")
    rankext_test_later = make_eval_dataset(later_step_classes, name="final_later_steps")
    rankext_test_seen  = make_eval_dataset(all_seen_classes, name="final_all_seen")

    rankext_final_first = trainer_rankext_final.evaluate(rankext_test_first)
    rankext_final_later = trainer_rankext_final.evaluate(rankext_test_later)
    rankext_final_seen  = trainer_rankext_final.evaluate(rankext_test_seen)

    rankext_final_evals[method_name] = {
        "first_step": rankext_final_first,
        "later_steps": rankext_final_later,
        "all_seen": rankext_final_seen,
    }

    print(f"\n[{method_name}] FINAL")
    print("first_step :", rankext_final_first)
    print("later_steps:", rankext_final_later)
    print("all_seen   :", rankext_final_seen)

    calc_final_seen = (
        0.2 * float(rankext_final_first.get("eval_accuracy", np.nan))
        + 0.8 * float(rankext_final_later.get("eval_accuracy", np.nan))
    )
    print(
        f"[{method_name}] FINAL sanity | "
        f"all_seen={float(rankext_final_seen.get('eval_accuracy', np.nan)):.4f} | "
        f"recomputed≈{calc_final_seen:.4f}"
    )

## 9) Baseline: full fine-tune instead of LoRA (Step 2)

In [ ]:
ft_results = []

for s in range(2, num_steps + 1):
    stale_train_dir = f"outputs/{RUN_NAME}/step_{s}_ft"
    stale_final_dir = f"outputs/{RUN_NAME}/step_{s}_ft_final"
    if os.path.exists(stale_train_dir):
        shutil.rmtree(stale_train_dir)
    if os.path.exists(stale_final_dir):
        shutil.rmtree(stale_final_dir)

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    print(f"\n[FT] Step {step_idx+1}")
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    tr_min, tr_max = _label_range(train_step)
    te_min, te_max = _label_range(test_step)

    print("Train label range:", tr_min, tr_max)
    print("Test label range:", te_min, te_max)
    print("Num labels:", num_classes)

    print("Loaded base_model_dir:", base_model_dir)
    ft_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    print("Loaded FT model num_labels:", ft_model.config.num_labels)
    print("Loaded FT classifier out_features:", ft_model.classifier.out_features)

    assert ft_model.config.num_labels == num_classes, (
        f"Loaded FT model num_labels={ft_model.config.num_labels}, expected {num_classes}"
    )
    assert ft_model.classifier.out_features == num_classes, (
        f"Loaded FT classifier out_features={ft_model.classifier.out_features}, expected {num_classes}"
    )

    assert tr_min >= 0
    assert tr_max < ft_model.classifier.out_features, (
        f"Train labels out of range: max label {tr_max}, classifier out_features {ft_model.classifier.out_features}"
    )
    assert te_min >= 0
    assert te_max < ft_model.classifier.out_features, (
        f"Test labels out of range: max label {te_max}, classifier out_features {ft_model.classifier.out_features}"
    )

    step_ft_train_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft"

    args_ft = TrainingArguments(
        output_dir=step_ft_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=FT_EPOCHS,
        learning_rate=LR_FT,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft = Trainer(
        model=ft_model,
        args=args_ft,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_ft.train()
    eval_current = trainer_ft.evaluate(test_step)

    pd.DataFrame(trainer_ft.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_log_history.csv"),
        index=False
    )

    step_ft_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_final"
    os.makedirs(step_ft_dir, exist_ok=True)

    print(f"[FT] Step {step_idx+1} saving model to {step_ft_dir}")
    trainer_ft.save_model(step_ft_dir)
    image_processor.save_pretrained(step_ft_dir)
    print(f"[FT] Step {step_idx+1} save finished")

    reloaded_ft = AutoModelForImageClassification.from_pretrained(step_ft_dir)
    print(f"[FT] Reload check step {step_idx+1} num_labels:", reloaded_ft.config.num_labels)
    print(f"[FT] Reload check step {step_idx+1} classifier out_features:", reloaded_ft.classifier.out_features)

    assert reloaded_ft.config.num_labels == num_classes, (
        f"Reloaded saved FT num_labels={reloaded_ft.config.num_labels}, expected {num_classes}"
    )
    assert reloaded_ft.classifier.out_features == num_classes, (
        f"Reloaded saved FT classifier out_features={reloaded_ft.classifier.out_features}, expected {num_classes}"
    )

    base_model_dir = step_ft_dir

    seen_classes_now = classes_for_cumulative(step_idx)
    eval_first = make_eval_dataset(first_step_classes, name=f"ft_step{step_idx+1}_first_step")

    later_seen_now = [c for c in seen_classes_now if c not in first_step_classes]
    eval_later = make_eval_dataset(later_seen_now, name=f"ft_step{step_idx+1}_later_seen") if len(later_seen_now) > 0 else None
    eval_seen = make_eval_dataset(seen_classes_now, name=f"ft_step{step_idx+1}_all_seen")

    metrics_first = trainer_ft.evaluate(eval_first)
    metrics_seen = trainer_ft.evaluate(eval_seen)

    if eval_later is not None:
        metrics_later = trainer_ft.evaluate(eval_later)
    else:
        metrics_later = {"eval_accuracy": np.nan, "eval_loss": np.nan}

    ft_results.extend([
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(metrics_first.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_first.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "later_steps_seen_so_far",
            "eval_accuracy": float(metrics_later.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_later.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(metrics_seen.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_seen.get("eval_loss", np.nan)),
        },
    ])

print("\nFull FT continual training finished.")

In [ ]:
final_ft_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_final"
final_ft_model = AutoModelForImageClassification.from_pretrained(final_ft_dir)

args_ft_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_final = Trainer(
    model=final_ft_model,
    args=args_ft_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

ft_test_first = make_eval_dataset(first_step_classes)
ft_test_later = make_eval_dataset(later_step_classes)
ft_test_seen = make_eval_dataset(all_seen_classes)

print("Final FT num_labels:", final_ft_model.config.num_labels)
print("Final FT classifier out_features:", final_ft_model.classifier.out_features)
assert final_ft_model.classifier.out_features == num_classes

ft_final_first = trainer_ft_final.evaluate(ft_test_first)
ft_final_later = trainer_ft_final.evaluate(ft_test_later)
ft_final_seen = trainer_ft_final.evaluate(ft_test_seen)

print("FT final - first step:", ft_final_first)
print("FT final - later steps:", ft_final_later)
print("FT final - all seen:", ft_final_seen)

## 10) Upper bound: joint training (full dataset)

In [ ]:
train_joint, test_joint, label2new_J, new2orig_J, class_ids_J = make_step_datasets(
    step_idx=0, split_type="full", remap_labels=False
)

config_joint = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

joint_model = AutoModelForImageClassification.from_config(config_joint)

args_joint = TrainingArguments(
    output_dir=JOINT_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=JOINT_EPOCHS,
    learning_rate=LR_JOINT,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_FT,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_FT,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer_joint = Trainer(
    model=joint_model,
    args=args_joint,
    train_dataset=train_joint,
    eval_dataset=test_joint,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer_joint.train()

eval_joint = trainer_joint.evaluate()
print({"eval_joint_full": eval_joint})

first_step_classes = classes_for_step(0)

later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(classes_for_step(s))

all_seen_classes = classes_for_cumulative(num_steps - 1)

joint_eval_first = make_eval_dataset(first_step_classes, name="joint_first")
joint_eval_later = make_eval_dataset(later_step_classes, name="joint_later")
joint_eval_seen  = make_eval_dataset(all_seen_classes, name="joint_seen")

joint_final_first = trainer_joint.evaluate(eval_dataset=joint_eval_first)
joint_final_later = trainer_joint.evaluate(eval_dataset=joint_eval_later)
joint_final_seen  = trainer_joint.evaluate(eval_dataset=joint_eval_seen)

print("Joint final - first_step:", joint_final_first)
print("Joint final - later_steps:", joint_final_later)
print("Joint final - all_seen:", joint_final_seen)

In [ ]:
joint_log_df = pd.DataFrame(trainer_joint.state.log_history)
joint_log_df.to_csv(os.path.join(TABLES_DIR, "joint_log_history.csv"), index=False)

joint_metrics = {
    "experiment": "joint_full",
    "eval_loss": float(eval_joint.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_joint.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "joint_metrics.json"), "w") as f:
    json.dump(joint_metrics, f, indent=2)

joint_log_df.tail()

In [ ]:
joint_test_first = make_eval_dataset(first_step_classes)
joint_test_later = make_eval_dataset(later_step_classes)
joint_test_all = make_eval_dataset(all_classes)

joint_final_first = trainer_joint.evaluate(joint_test_first)
joint_final_later = trainer_joint.evaluate(joint_test_later)
joint_final_all = trainer_joint.evaluate(joint_test_all)

print("Joint final - first step:", joint_final_first)
print("Joint final - later steps:", joint_final_later)
print("Joint final - all classes:", joint_final_all)

## 11) Compare results (step test vs full test)

In [ ]:
def grab_acc(d):
    return float(d["eval_accuracy"]) if "eval_accuracy" in d else np.nan

def grab_loss(d):
    return float(d["eval_loss"]) if "eval_loss" in d else np.nan

all_results = []

# step-wise rankext
all_results.extend(rankext_results)

# step-wise ft
all_results.extend(ft_results)

# final rankext
for method_name, evals_dict in rankext_final_evals.items():
    exp_tag = "rankext_ortho" if method_name == "RankExt+Ortho" else "rankext_no_ortho"
    for eval_type, res in evals_dict.items():
        all_results.append({
            "experiment": f"{exp_tag}_final_eval",
            "method": method_name,
            "step": num_steps,
            "eval_type": eval_type,
            "eval_accuracy": grab_acc(res),
            "eval_loss": grab_loss(res),
        })

# final ft
all_results.extend([
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(ft_final_first),
        "eval_loss": grab_loss(ft_final_first),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(ft_final_later),
        "eval_loss": grab_loss(ft_final_later),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(ft_final_seen),
        "eval_loss": grab_loss(ft_final_seen),
    },
])

# final joint
all_results.extend([
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(joint_final_first),
        "eval_loss": grab_loss(joint_final_first),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(joint_final_later),
        "eval_loss": grab_loss(joint_final_later),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(joint_final_seen),
        "eval_loss": grab_loss(joint_final_seen),
    },
])

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(TABLES_DIR, "results_raw.csv"), index=False)

print(results_df)

In [ ]:
results_df_clean = results_df.copy()

results_df_clean = results_df_clean.rename(columns={
    "eval_accuracy": "accuracy",
    "eval_loss": "loss",
    "eval_type": "eval_set"
})

def map_method(x):
    if x == "step1_scratch":
        return "B0"
    elif x == "RankExt+Ortho":
        return "RankExt+Ortho"
    elif x == "RankExt":
        return "RankExt"
    elif x == "rank_extension":
        return "RankExt"
    elif x == "full_finetune":
        return "Full FT"
    elif x == "joint_upper_bound":
        return "Joint"
    return x

results_df_clean["method"] = results_df_clean["method"].apply(map_method)

def map_eval_stage(exp_name):
    if "final_eval" in str(exp_name):
        return "final"
    return "step"

results_df_clean["eval_stage"] = results_df_clean["experiment"].apply(map_eval_stage)

def map_adapter_name(row):
    if row["method"] == "B0":
        return "B0"
    if row["method"] in ["RankExt", "RankExt+Ortho"]:
        step = int(row["step"])
        if row["eval_stage"] == "final":
            return f"{row['method']}_final"
        return f"{row['method']}_s{step}"
    if row["method"] == "Full FT":
        return f"FT_s{int(row['step'])}" if row["eval_stage"] == "step" else "FT_final"
    if row["method"] == "Joint":
        return "Joint_final"
    return np.nan

results_df_clean["adapter_name"] = results_df_clean.apply(map_adapter_name, axis=1)

results_df_clean = results_df_clean[
    ["method", "adapter_name", "step", "eval_stage", "eval_set", "accuracy", "loss"]
].copy()

results_df_clean = results_df_clean.sort_values(
    by=["method", "eval_stage", "step", "eval_set"]
).reset_index(drop=True)

dups = results_df_clean.duplicated(
    subset=["method", "step", "eval_stage", "eval_set"],
    keep=False
)
if dups.any():
    print("WARNING: duplicate rows found:")
    print(results_df_clean[dups])
else:
    print("No duplicate rows found.")

results_df_clean.to_csv(os.path.join(TABLES_DIR, "results_summary_clean.csv"), index=False)

print(results_df_clean)

In [ ]:
summary_lines = [
    "Experiment summary",
    "==================",
    "",
]

for _, row in results_df_clean.iterrows():
    acc = row["accuracy"]
    loss = row["loss"]

    acc_str = f"{acc:.4f}" if pd.notna(acc) else "nan"
    loss_str = f"{loss:.4f}" if pd.notna(loss) else "nan"

    summary_lines.append(
        f"method={row['method']} | adapter_name={row['adapter_name']} | "
        f"step={row['step']} | eval_stage={row['eval_stage']} | "
        f"eval_set={row['eval_set']} | accuracy={acc_str} | loss={loss_str}"
    )

with open(os.path.join(RESULTS_DIR, "summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_final = results_df_clean[
    (results_df_clean["eval_stage"] == "final") &
    (results_df_clean["step"] == num_steps)
].copy()

df_final = df_final[df_final["method"].isin([
    "RankExt",
    "RankExt+Ortho",
    "Full FT",
    "Joint"
])]

labels = ["first_step", "later_steps", "all_seen"]

pivot = df_final.pivot_table(
    index="eval_set",
    columns="method",
    values="accuracy",
    aggfunc="mean"
).reindex(labels)

method_order = ["RankExt", "RankExt+Ortho", "Full FT", "Joint"]
pivot = pivot.reindex(columns=method_order)

x = np.arange(len(labels))
width = 0.2

plt.figure(figsize=(11, 5))
for i, method_name in enumerate(method_order):
    plt.bar(x + (i - 1.5) * width, pivot[method_name], width, label=method_name)

plt.xticks(x, labels)
plt.ylabel("Accuracy")
plt.title("Final Accuracy Comparison: RankExt vs RankExt+Ortho vs Full FT vs Joint")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "final_accuracy_comparison_rankext_ortho_ft_joint.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

In [ ]:
final_first = df_final[df_final["eval_set"] == "first_step"]
final_later = df_final[df_final["eval_set"] == "later_steps"]
final_seen  = df_final[df_final["eval_set"] == "all_seen"]

def make_barplot(df_sub, title, filename):
    methods = ["RankExt", "RankExt+Ortho", "Full FT", "Joint"]
    vals = []
    for m in methods:
        row = df_sub[df_sub["method"] == m]
        vals.append(float(row["accuracy"].iloc[0]) if len(row) > 0 else np.nan)

    plt.figure(figsize=(8, 4))
    plt.bar(methods, vals)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved plot to:", out_path)

make_barplot(final_first, "Final Accuracy on B0 Classes", "final_first_step_accuracy_rankext_ortho_ft_joint.png")
make_barplot(final_later, "Final Accuracy on Later Classes", "final_later_steps_accuracy_rankext_ortho_ft_joint.png")
make_barplot(final_seen,  "Final Accuracy on All Seen Classes", "final_all_seen_accuracy_rankext_ortho_ft_joint.png")

In [ ]:
plot_df_step = results_df_clean[results_df_clean["eval_stage"] == "step"].copy()


for stale_name in [
    "curve_rank_extension_current_step.png",
    "curve_rankext_current_step.png",
    "curve_rankext_plus_ortho_current_step.png",
]:
    stale_path = os.path.join(PLOTS_DIR, stale_name)
    if os.path.exists(stale_path):
        os.remove(stale_path)
        print("Removed stale plot:", stale_path)

for method_name in ["RankExt", "RankExt+Ortho"]:
    sub = plot_df_step[
        (plot_df_step["method"] == method_name) &
        (plot_df_step["eval_set"] == "current_step")
    ].copy()

    sub = sub.sort_values("step").reset_index(drop=True)

    dup_mask = sub.duplicated(subset=["method", "step", "eval_set"], keep=False)
    if dup_mask.any():
        print(f"WARNING: duplicates in current-step plot for {method_name}")
        print(sub[dup_mask])

    print(f"\nData used for {method_name} current-step plot:")
    print(sub[["method", "adapter_name", "step", "accuracy"]])

    plt.figure(figsize=(7, 4))
    plt.plot(sub["step"], sub["accuracy"], marker="o")
    plt.xticks(sub["step"], sub["adapter_name"])
    plt.xlabel("Step")
    plt.ylabel("Accuracy")
    plt.title(f"{method_name}: Current-Step Accuracy")
    plt.ylim(0, max(0.7, float(sub["accuracy"].max()) + 0.05))
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()

    out = os.path.join(
        PLOTS_DIR,
        f"curve_{method_name.lower().replace('+','_plus_')}_current_step_clean.png"
    )
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

In [ ]:
plot_df_step = results_df_clean[results_df_clean["eval_stage"] == "step"].copy()

plt.figure(figsize=(8, 4))

for method_name in ["RankExt", "RankExt+Ortho", "Full FT"]:
    sub = plot_df_step[
        (plot_df_step["method"] == method_name) &
        (plot_df_step["eval_set"] == "first_step")
    ].copy()

    if len(sub) > 0:
        sub = sub.sort_values("step").reset_index(drop=True)
        plt.plot(sub["step"], sub["accuracy"], marker="o", label=method_name)

plt.xlabel("Step")
plt.ylabel("Accuracy on B0 classes")
plt.title("Retention on First-Step Classes Across Continual Steps")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()

out = os.path.join(PLOTS_DIR, "curve_retention_first_step_across_steps.png")
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", out)